# Holt_Winters — otimização walk-forward

A seleção usa apenas os 80% de desenvolvimento. Em cada origem de validação o modelo é reajustado e prevê somente a próxima hora.

**Esta etapa é pesada e possui checkpoint em CSV.** Se interromper, rode novamente para continuar dos candidatos já concluídos.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
ranking_path = config.RESULT_DIR / "tuning_candidates_Holt_Winters.csv"

winner, ranking = models.tune_holt_winters(model_df, ranking_path)
save_json(winner, config.RESULT_DIR / "best_params_Holt_Winters.json")
all_params_path = config.RESULT_DIR / "best_params.json"
all_params = load_json(all_params_path) if all_params_path.exists() else {}
all_params["Holt_Winters"] = winner
save_json(all_params, all_params_path)
print("Vencedor:", winner)
display(ranking)



Holt_Winters candidato 1/18: {'trend': None, 'seasonal': 'add', 'damped_trend': False, 'seasonal_periods': 24, 'max_train_rows': 1440}
Holt_Winters: 20/258 origens | MAE parcial=230.850
Holt_Winters: 40/258 origens | MAE parcial=182.075
Holt_Winters: 60/258 origens | MAE parcial=181.467
Holt_Winters: 80/258 origens | MAE parcial=200.113
Holt_Winters: 100/258 origens | MAE parcial=199.690
Holt_Winters: 120/258 origens | MAE parcial=324.792
Holt_Winters: 140/258 origens | MAE parcial=321.843
Holt_Winters: 160/258 origens | MAE parcial=318.750
Holt_Winters: 180/258 origens | MAE parcial=359.822
Holt_Winters: 200/258 origens | MAE parcial=364.170
Holt_Winters: 220/258 origens | MAE parcial=356.259
Holt_Winters: 240/258 origens | MAE parcial=381.467
Holt_Winters: 258/258 origens | MAE parcial=399.578

Holt_Winters candidato 2/18: {'trend': 'add', 'seasonal': 'add', 'damped_trend': False, 'seasonal_periods': 24, 'max_train_rows': 1440}
Holt_Winters: 20/258 origens | MAE parcial=230.850
Holt

,model,parameters,validation_mae,mean_seconds_per_origin,elapsed_seconds,convergence_rate,status,protocol_signature,component_complexity,period_complexity,window_complexity,mae_3
0,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 1440...",349.463011,0.038530,9.940613,0.507752,ok,332653fdf1d85ebc,1,168,1440,349.463
1,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 2880...",371.203493,0.051263,13.225798,0.127907,ok,332653fdf1d85ebc,1,168,2880,371.203
2,Holt_Winters,"{""damped_trend"": true, ""max_train_rows"": 1440,...",391.699453,0.183486,47.339394,0.139535,ok,332653fdf1d85ebc,3,168,1440,391.699
3,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 1440...",392.299840,0.179412,46.288367,0.120155,ok,332653fdf1d85ebc,2,168,1440,392.300
4,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 1440...",399.577519,0.022090,5.699133,0.000000,ok,332653fdf1d85ebc,1,24,1440,399.578
5,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 2880...",399.577519,0.026448,6.823712,0.000000,ok,332653fdf1d85ebc,1,24,2880,399.578
6,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 4320...",399.577519,0.035327,9.114448,0.000000,ok,332653fdf1d85ebc,1,24,4320,399.578
7,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 4320...",399.577519,0.061818,15.949034,0.000000,ok,332653fdf1d85ebc,1,168,4320,399.578
8,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 1440...",399.577519,0.167865,43.309045,0.000000,ok,332653fdf1d85ebc,2,24,1440,399.578
9,Holt_Winters,"{""damped_trend"": false, ""max_train_rows"": 2880...",399.577519,0.299412,77.248290,0.000000,ok,332653fdf1d85ebc,2,24,2880,399.578


O teste final não foi usado nesta escolha. Revise o ranking antes de executar o notebook de walk-forward final.